In [1]:
from streaming_helper_chat import add_assistant_message, add_user_message, chat_stream
from streaming_helper_tools import save_article_schema, save_article

Python: /Users/msingh/CodeHub/all-about-claude/.venv/bin/python
API key loaded: True


In [2]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [3]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [4]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
)


>>> Tool Call: "save_article"
{"abstract": "A novel deep learning architecture for optimizing recursive neural networks through quantum-inspired gradient descent.", "meta": {
  "word_count": 4250,
  "review": "This paper presents an innovative approach to neural network optimization by combining recursive architectures with quantum-inspired algorithms. The authors demonstrate strong empirical results on benchmark datasets, showing a 23% improvement over baseline methods. The theoretical framework is well-motivated and the mathematical derivations are rigorous. However, the computational complexity analysis could be more thorough, and the practical applicability to large-scale systems remains unclear. The experimental section covers diverse scenarios but lacks comparison with some recent state-of-the-art methods. The writing is generally clear, though some sections could benefit from simplification for broader accessibility. Overall, this represents a solid contribution to the field of

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01BRjhC145zW2yc22un8rym6',
    'name': 'save_article',
    'input': {'abstract': 'A novel deep learning architecture for optimizing recursive neural networks through quantum-inspired gradient descent.',
     'meta': {'word_count': 4250,
      'review': 'This paper presents an innovative approach to neural network optimization by combining recursive architectures with quantum-inspired algorithms. The authors demonstrate strong empirical results on benchmark datasets, showing a 23% improvement over baseline methods. The theoretical framework is well-motivated and the mathematical derivations are rigorous. However, the computational complexity analysis could be more thorough, and the practical applicability to large-scale systems remains unclear. The experimental section covers diverse scenarios but la